# Padrões de erro nas questões de Física do ENEM 2024
**Análise de distratores a partir dos microdados do INEP**

Rode as células **de cima para baixo, uma vez cada** (`Ambiente de execução → Executar tudo`).

| Etapa | O que faz | Tempo aproximado |
|---|---|---|
| 0 | Monta o Drive e define as configurações | segundos |
| 1 | Converte o CSV de resultados para Parquet (**só na primeira vez**) | 5–15 min |
| 2 | Lê os itens, identifica as questões de Física e mapeia as cores para o caderno azul | segundos |
| 3 | Carrega as respostas e padroniza tudo na ordem do caderno azul | 1–2 min |
| 4 | Taxa de acerto e frequência de cada alternativa (distratores) | segundos |
| 5 | Curvas de escolha das alternativas em três réguas de desempenho | 1–2 min |
| 6 | Parâmetros a, b, c da TRI e acerto previsto × observado | segundos |
| 7 | Exporta tabelas e figuras para `processed/` | segundos |

Se o Colab reiniciar no meio, basta rodar de novo: a Etapa 1 detecta o Parquet já criado e é pulada.

In [ ]:
# ============================================================
# ETAPA 0 — DRIVE E CONFIGURAÇÕES
# ============================================================
import os, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # fora do Colab

# Pasta com os microdados (ajuste para o seu Drive)
PASTA_PROJETO = os.environ.get("PASTA_PROJETO", "/content/drive/MyDrive/TCC Melissa/enem-fisica")
PASTA_RAW   = os.path.join(PASTA_PROJETO, "raw")        # RESULTADOS_2024.csv e ITENS_PROVA_2024.csv
PASTA_SAIDA = os.path.join(PASTA_PROJETO, "processed")  # resultados gerados
PASTA_FIG   = os.path.join(PASTA_SAIDA, "figuras")
os.makedirs(PASTA_FIG, exist_ok=True)

def localizar(nome):
    """Aceita o nome original do INEP ou uma cópia ("Cópia de ...")."""
    for candidato in [nome, f"Cópia de {nome}"]:
        caminho = os.path.join(PASTA_RAW, candidato)
        if os.path.isfile(caminho):
            return caminho
    raise FileNotFoundError(f"{nome} não encontrado em {PASTA_RAW}: {sorted(os.listdir(PASTA_RAW))}")

ARQ_RESULTADOS = localizar("RESULTADOS_2024.csv")
ARQ_ITENS      = localizar("ITENS_PROVA_2024.csv")
ARQ_PARQUET    = os.path.join(PASTA_SAIDA, "resultados_cn_2024.parquet")

# Cadernos regulares de CN da aplicação principal (azul = referência)
CO_PROVA_AZUL = 1419
CODIGOS_REGULARES = [1419, 1420, 1421, 1422]

# Questões de Física na numeração do caderno AZUL (Quadro 1 do TCC)
QUESTOES_FISICA = [91, 96, 99, 100, 101, 103, 106,
                   119, 121, 122, 127, 129, 130, 131, 132]

COL_PROFICIENCIA = "NU_NOTA_CN"   # régua de comparação (nota TRI)
N_FAIXAS = 20
PRIMEIRA_POSICAO_CN = 91          # CN ocupa as posições 91–135

print("Resultados:", ARQ_RESULTADOS)
print("Itens:     ", ARQ_ITENS)

In [ ]:
# ============================================================
# ETAPA 1 — CSV → PARQUET (só colunas necessárias, em blocos)
# ============================================================
# Ler o CSV inteiro (42 colunas × 4,3 milhões de linhas) estoura a
# memória do Colab. Aqui ele é lido em blocos de 500 mil linhas, já
# filtrando presença nas 4 áreas e os cadernos regulares de CN.

COLUNAS_LER = [
    "CO_PROVA_CN",
    "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT",
    "TX_RESPOSTAS_CN",
]

cabecalho = pd.read_csv(ARQ_RESULTADOS, sep=";", encoding="latin1", nrows=0)
cabecalho = [c.strip() for c in cabecalho.columns]
faltantes = [c for c in COLUNAS_LER if c not in cabecalho]
if faltantes:
    raise ValueError(f"Colunas ausentes em RESULTADOS: {faltantes}")

# Treineiros: só é possível excluir se a coluna existir neste arquivo.
TEM_TREINEIRO = "IN_TREINEIRO" in cabecalho
if TEM_TREINEIRO:
    COLUNAS_LER.append("IN_TREINEIRO")
print("Coluna IN_TREINEIRO em RESULTADOS:", TEM_TREINEIRO)

if os.path.isfile(ARQ_PARQUET):
    print("Parquet já existe — etapa pulada:", ARQ_PARQUET)
else:
    contagem = {"lidos": 0, "presentes_4_areas": 0, "regulares": 0, "gravados": 0}
    writer = None
    leitor = pd.read_csv(
        ARQ_RESULTADOS, sep=";", encoding="latin1",
        usecols=lambda c: c.strip() in COLUNAS_LER,
        dtype={"TX_RESPOSTAS_CN": "string"},
        chunksize=500_000,
    )
    for i, bloco in enumerate(leitor, start=1):
        bloco.columns = bloco.columns.str.strip()
        contagem["lidos"] += len(bloco)

        for c in ["CO_PROVA_CN", "TP_PRESENCA_CN", "TP_PRESENCA_CH",
                  "TP_PRESENCA_LC", "TP_PRESENCA_MT"]:
            bloco[c] = pd.to_numeric(bloco[c], errors="coerce")

        presentes = (bloco["TP_PRESENCA_CN"].eq(1) & bloco["TP_PRESENCA_CH"].eq(1)
                     & bloco["TP_PRESENCA_LC"].eq(1) & bloco["TP_PRESENCA_MT"].eq(1))
        contagem["presentes_4_areas"] += int(presentes.sum())
        bloco = bloco.loc[presentes]

        bloco = bloco.loc[bloco["CO_PROVA_CN"].isin(CODIGOS_REGULARES)]
        contagem["regulares"] += len(bloco)

        if TEM_TREINEIRO:
            bloco = bloco.loc[pd.to_numeric(bloco["IN_TREINEIRO"], errors="coerce").ne(1)]

        saida = pd.DataFrame({
            "CO_PROVA_CN": bloco["CO_PROVA_CN"].astype("int16"),
            "NU_NOTA_CN": pd.to_numeric(bloco["NU_NOTA_CN"], errors="coerce").astype("float32"),
            "NU_NOTA_CH": pd.to_numeric(bloco["NU_NOTA_CH"], errors="coerce").astype("float32"),
            "NU_NOTA_LC": pd.to_numeric(bloco["NU_NOTA_LC"], errors="coerce").astype("float32"),
            "NU_NOTA_MT": pd.to_numeric(bloco["NU_NOTA_MT"], errors="coerce").astype("float32"),
            "TX_RESPOSTAS_CN": bloco["TX_RESPOSTAS_CN"].fillna("").str.strip().astype(str),
        })
        tabela = pa.Table.from_pandas(saida, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(ARQ_PARQUET, tabela.schema, compression="snappy")
        writer.write_table(tabela)
        contagem["gravados"] += len(saida)
        print(f"Bloco {i}: {contagem['lidos']:,} linhas lidas")
        del bloco, saida, tabela; gc.collect()

    if writer is not None:
        writer.close()
    print()
    for k, v in contagem.items():
        print(f"{k:>20}: {v:,}")

In [ ]:
# ============================================================
# ETAPA 2 — ITENS, GABARITOS, TRI E MAPEAMENTO PARA O AZUL
# ============================================================
itens = pd.read_csv(ARQ_ITENS, sep=";", encoding="latin1", low_memory=False)
itens.columns = itens.columns.str.strip()

for c in ["CO_POSICAO", "CO_PROVA", "CO_ITEM", "CO_HABILIDADE", "IN_ITEM_ABAN",
          "NU_PARAM_A", "NU_PARAM_B", "NU_PARAM_C"]:
    if c in itens.columns:
        itens[c] = pd.to_numeric(itens[c], errors="coerce")
for c in ["SG_AREA", "TX_COR", "TX_GABARITO"]:
    itens[c] = itens[c].astype("string").str.strip().str.upper()

itens_cn = itens.loc[itens["SG_AREA"].eq("CN") & itens["CO_PROVA"].isin(CODIGOS_REGULARES)].copy()

print("Cadernos de CN usados:")
print(itens_cn.groupby(["CO_PROVA", "TX_COR"]).size().rename("itens").to_string())

azul = itens_cn.loc[itens_cn["CO_PROVA"].eq(CO_PROVA_AZUL)].set_index("CO_POSICAO")
if len(azul) != 45:
    raise ValueError(f"O caderno {CO_PROVA_AZUL} tem {len(azul)} itens (esperado 45).")
if azul["TX_COR"].iloc[0] != "AZUL":
    raise ValueError(f"O código {CO_PROVA_AZUL} não é AZUL no arquivo de itens.")

# Mapa: para cada caderno, em que posição (0–44) da string de respostas
# está o item que ocupa cada posição do caderno azul.
itens_azul_ordem = azul.sort_index()["CO_ITEM"].to_numpy()
MAPA_COR_PARA_AZUL = {}
for codigo, grupo in itens_cn.groupby("CO_PROVA"):
    pos_por_item = dict(zip(grupo["CO_ITEM"], grupo["CO_POSICAO"] - PRIMEIRA_POSICAO_CN))
    faltam = [it for it in itens_azul_ordem if it not in pos_por_item]
    if faltam:
        raise ValueError(f"Caderno {codigo} não contém os itens {faltam} do azul.")
    MAPA_COR_PARA_AZUL[int(codigo)] = np.array([pos_por_item[it] for it in itens_azul_ordem])
print("\nMapeamento por CO_ITEM construído para:", sorted(MAPA_COR_PARA_AZUL))

# Tabela das questões de Física (numeração do azul)
colunas_itens = ["CO_ITEM", "CO_HABILIDADE", "TX_GABARITO", "IN_ITEM_ABAN",
                 "TX_MOTIVO_ABAN", "NU_PARAM_A", "NU_PARAM_B", "NU_PARAM_C"]
fisica = (azul.loc[QUESTOES_FISICA, [c for c in colunas_itens if c in azul.columns]]
          .rename_axis("Questao").reset_index())
fisica["Situacao"] = np.where(fisica["IN_ITEM_ABAN"].eq(1), "Anulada",
                     np.where(fisica["TX_GABARITO"].isin(list("ABCDE")), "Válida",
                              "Verificar gabarito"))
if fisica["Situacao"].eq("Verificar gabarito").any():
    display(fisica)
    raise ValueError("Há item sem gabarito válido e sem indicação de anulação.")

# Gabarito igual em todas as cores? (checagem de consistência)
gab_por_cor = itens_cn.loc[itens_cn["CO_ITEM"].isin(fisica["CO_ITEM"])].groupby("CO_ITEM")["TX_GABARITO"].nunique()
if (gab_por_cor > 1).any():
    print("ATENÇÃO: itens com gabarito diferente entre cores:", list(gab_por_cor[gab_por_cor > 1].index))

display(fisica)

In [ ]:
# ============================================================
# ETAPA 3 — RESPOSTAS NA ORDEM DO CADERNO AZUL
# ============================================================
base = pd.read_parquet(ARQ_PARQUET)
print("Participantes na base (presentes nas 4 áreas, cadernos regulares):", f"{len(base):,}")

tam = base["TX_RESPOSTAS_CN"].str.len()
if (tam != 45).any():
    print(f"Removidos {int((tam != 45).sum()):,} registros com string de respostas ≠ 45 caracteres")
    base = base.loc[tam.eq(45)].reset_index(drop=True)

# Matriz n × 45 de bytes (rápido e econômico: ~140 MB para 3 milhões)
bruto = np.frombuffer("".join(base["TX_RESPOSTAS_CN"]).encode("ascii"), dtype=np.uint8)
resp = bruto.reshape(len(base), 45)
base = base.drop(columns="TX_RESPOSTAS_CN")
del bruto; gc.collect()

idx_fisica = [q - PRIMEIRA_POSICAO_CN for q in QUESTOES_FISICA]
codigos = base["CO_PROVA_CN"].to_numpy()

# Só as colunas de Física, já reordenadas para o azul
resp_fisica = np.empty((len(base), len(QUESTOES_FISICA)), dtype=np.uint8)
for codigo, mapa in MAPA_COR_PARA_AZUL.items():
    linhas = codigos == codigo
    resp_fisica[linhas] = resp[linhas][:, mapa[idx_fisica]]
del resp; gc.collect()

respostas_fisica = pd.DataFrame(
    resp_fisica.view("S1").astype(str), columns=[f"Q{q}" for q in QUESTOES_FISICA]
)
respostas_fisica["CO_PROVA_CN"] = codigos
respostas_fisica[COL_PROFICIENCIA] = base[COL_PROFICIENCIA].to_numpy()
N_PARTICIPANTES = len(respostas_fisica)

print("Participantes por caderno:")
print(respostas_fisica["CO_PROVA_CN"].value_counts().sort_index().to_string())
print("\nCaracteres encontrados nas respostas de Física:",
      sorted(pd.unique(respostas_fisica[[f'Q{q}' for q in QUESTOES_FISICA]].to_numpy().ravel())))
display(respostas_fisica.head())

In [ ]:
# ============================================================
# ETAPA 4 — TAXA DE ACERTO E FREQUÊNCIA DAS ALTERNATIVAS
# ============================================================
# '.' = em branco; '*' = dupla marcação (padrão do INEP). Ambos contam como erro.
ROTULOS = {"A": "A", "B": "B", "C": "C", "D": "D", "E": "E", ".": "Branco", "*": "Dupla"}
ALTERNATIVAS = list("ABCDE")

linhas = []
freq_longa = []
for _, it in fisica.iterrows():
    q = int(it["Questao"])
    col = respostas_fisica[f"Q{q}"].map(ROTULOS).fillna("Outro")
    pct = col.value_counts(normalize=True).mul(100)
    linha = {"Questao": q, "CO_ITEM": it["CO_ITEM"], "Habilidade": it["CO_HABILIDADE"],
             "Gabarito": it["TX_GABARITO"], "Situacao": it["Situacao"],
             "Param_a": it.get("NU_PARAM_A"), "Param_b": it.get("NU_PARAM_B"),
             "Param_c": it.get("NU_PARAM_C")}
    for alt in ALTERNATIVAS + ["Branco", "Dupla", "Outro"]:
        linha[f"%{alt}"] = pct.get(alt, 0.0)
    if it["Situacao"] == "Válida":
        gab = it["TX_GABARITO"]
        linha["Taxa_Acerto_%"] = linha[f"%{gab}"]
        distr = {a: linha[f"%{a}"] for a in ALTERNATIVAS if a != gab}
        top = max(distr, key=distr.get)
        linha["Distrator_mais_escolhido"] = top
        linha["%Distrator_mais_escolhido"] = distr[top]
        linha["Distrator_supera_gabarito"] = distr[top] > linha["Taxa_Acerto_%"]
    linhas.append(linha)

tabela_distratores = pd.DataFrame(linhas)
validas = tabela_distratores.loc[tabela_distratores["Situacao"].eq("Válida")].copy()

print(f"Participantes analisados: {N_PARTICIPANTES:,}")
print(f"Questões válidas: {len(validas)} | anuladas: {int(tabela_distratores['Situacao'].eq('Anulada').sum())}")
cols_show = ["Questao", "Gabarito", "Taxa_Acerto_%"] + [f"%{a}" for a in ALTERNATIVAS] + \
            ["%Branco", "Distrator_mais_escolhido", "%Distrator_mais_escolhido", "Distrator_supera_gabarito"]
display(tabela_distratores[cols_show].style.format(precision=1))

estatisticas = validas["Taxa_Acerto_%"].describe().rename({
    "count": "Questões válidas", "mean": "Média (%)", "std": "Desvio-padrão (p.p.)",
    "min": "Mínimo (%)", "25%": "1º quartil (%)", "50%": "Mediana (%)",
    "75%": "3º quartil (%)", "max": "Máximo (%)"}).to_frame("Valor")
display(estatisticas.style.format(precision=2))

# Gráfico de barras empilhadas: distribuição das respostas por questão
fig, ax = plt.subplots(figsize=(12, 6))
ordem = validas.sort_values("Taxa_Acerto_%")
esquerda = np.zeros(len(ordem))
for alt in ALTERNATIVAS + ["Branco"]:
    vals = ordem[f"%{alt}"].to_numpy()
    cores = ["#2a9d8f" if g == alt else "#c9c9c9" for g in ordem["Gabarito"]] if alt in ALTERNATIVAS else ["#ffffff"] * len(ordem)
    ax.barh(ordem["Questao"].astype(str), vals, left=esquerda, color=cores, edgecolor="white")
    for y, (v, l) in enumerate(zip(vals, esquerda)):
        if v >= 4:
            ax.text(l + v / 2, y, alt if alt in ALTERNATIVAS else "", ha="center", va="center", fontsize=8)
    esquerda += vals
ax.set_xlabel("% dos participantes"); ax.set_ylabel("Questão (caderno azul)")
ax.set_title("Distribuição das respostas — gabarito em verde, distratores em cinza")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIG, "distribuicao_respostas_fisica.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# ETAPA 5 — CURVAS DOS DISTRATORES × DESEMPENHO (TRÊS RÉGUAS)
# ============================================================
# Régua 1 — Nota TRI (NU_NOTA_CN): a nota oficial do INEP. Limitação: ela
#   é calculada com todas as respostas, inclusive a da questão analisada.
# Régua 2 — Acertos nos DEMAIS itens de CN: para cada questão, conta os
#   acertos do participante nos outros itens válidos de CN, excluindo a
#   própria questão (o item analisado não influencia a medida).
# Régua 3 — Acertos nas DEMAIS questões de FÍSICA (teste de robustez):
#   mais específica para Física, porém menos precisa (poucos itens).
# As três versões são geradas para comparação.

FAIXA_MIN_PARTICIPANTES = 1000   # pontos com menos gente que isso não são plotados
CODIGOS_RESPOSTA = {ord(a): i for i, a in enumerate("ABCDE")}  # A=0 ... E=4; resto=5

# ---------- Matriz completa de respostas de CN na ordem do azul ----------
base_cn = pd.read_parquet(ARQ_PARQUET, columns=["CO_PROVA_CN", "NU_NOTA_CN", "TX_RESPOSTAS_CN"])
base_cn = base_cn.loc[base_cn["TX_RESPOSTAS_CN"].str.len().eq(45)].reset_index(drop=True)
bruto = np.frombuffer("".join(base_cn["TX_RESPOSTAS_CN"]).encode("ascii"), dtype=np.uint8)
resp_cn = bruto.reshape(len(base_cn), 45)
cod_cn = base_cn["CO_PROVA_CN"].to_numpy()
nota_cn = base_cn["NU_NOTA_CN"].to_numpy(dtype="float64")
base_cn = base_cn.drop(columns="TX_RESPOSTAS_CN")
del bruto; gc.collect()

resp_azul = np.empty_like(resp_cn)
for codigo, mapa in MAPA_COR_PARA_AZUL.items():
    linhas = cod_cn == codigo
    resp_azul[linhas] = resp_cn[linhas][:, mapa]
del resp_cn; gc.collect()

# Gabarito do azul (posições 91–135) e itens válidos (não anulados)
azul_ord = azul.sort_index()
gab_azul = np.array([ord(g) if g in "ABCDE" else 0 for g in azul_ord["TX_GABARITO"].fillna("X")],
                    dtype=np.uint8)
valido_azul = (azul_ord["IN_ITEM_ABAN"].fillna(0).to_numpy() != 1) & (gab_azul != 0)
N_ITENS_VALIDOS_CN = int(valido_azul.sum())

acertos = (resp_azul == gab_azul) & valido_azul        # n × 45 (bool)
total_acertos_cn = acertos.sum(axis=1).astype(np.int16)
print(f"Participantes: {len(total_acertos_cn):,}")
print(f"Itens válidos de CN usados na régua de acertos: {N_ITENS_VALIDOS_CN}")
print(f"Média de acertos em CN: {total_acertos_cn.mean():.1f}")

# Régua 3: itens válidos de Física (posições no azul)
idx_fisica_validas = [int(q) - PRIMEIRA_POSICAO_CN for q in validas["Questao"]]
N_ITENS_VALIDOS_FIS = len(idx_fisica_validas)
total_acertos_fis = acertos[:, idx_fisica_validas].sum(axis=1).astype(np.int16)
print(f"Itens válidos de Física usados na régua 3: {N_ITENS_VALIDOS_FIS}")
print(f"Média de acertos em Física: {total_acertos_fis.mean():.1f}")

# Código da alternativa marcada (0–4 = A–E; 5 = branco/dupla/outro)
tabela_codigo = np.full(256, 5, dtype=np.uint8)
for byte, i in CODIGOS_RESPOSTA.items():
    tabela_codigo[byte] = i

def proporcoes_por_grupo(grupo, escolha, n_grupos):
    # % de cada alternativa (A–E) em cada grupo, usando contagem rápida
    contagem = np.bincount(grupo * 6 + escolha, minlength=n_grupos * 6).reshape(n_grupos, 6)
    n = contagem.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        pct = 100 * contagem[:, :5] / n[:, None]
    return pct, n

# ---------- Régua 1: nota TRI em quantis ----------
tem_nota = np.isfinite(nota_cn) & (nota_cn > 0)
faixa_tri = pd.qcut(nota_cn[tem_nota], q=N_FAIXAS, labels=False, duplicates="drop").astype(np.int64)
n_faixas_tri = int(faixa_tri.max()) + 1
nota_media_faixa = np.bincount(faixa_tri, weights=nota_cn[tem_nota]) / np.bincount(faixa_tri)

curvas_tri, curvas_acertos, curvas_fisica = [], [], []
for _, it in validas.iterrows():
    q = int(it["Questao"])
    j = q - PRIMEIRA_POSICAO_CN
    escolha = tabela_codigo[resp_azul[:, j]].astype(np.int64)

    # Régua 1
    pct, n = proporcoes_por_grupo(faixa_tri, escolha[tem_nota], n_faixas_tri)
    df = pd.DataFrame(pct, columns=ALTERNATIVAS)
    df.insert(0, "Nota_media_faixa", nota_media_faixa)
    df.insert(0, "Participantes", n)
    df.insert(0, "Questao", q)
    curvas_tri.append(df)

    # Régua 2: acertos nos demais itens (exclui a própria questão)
    resto = (total_acertos_cn - acertos[:, j]).astype(np.int64)
    n_grupos = N_ITENS_VALIDOS_CN  # resto vai de 0 a N_ITENS_VALIDOS_CN - 1
    pct, n = proporcoes_por_grupo(resto, escolha, n_grupos)
    df = pd.DataFrame(pct, columns=ALTERNATIVAS)
    df.insert(0, "Acertos_demais_itens", np.arange(n_grupos))
    df.insert(0, "Participantes", n)
    df.insert(0, "Questao", q)
    curvas_acertos.append(df)

    # Régua 3: acertos nas demais questões de Física
    resto_f = (total_acertos_fis - acertos[:, j]).astype(np.int64)
    pct, n = proporcoes_por_grupo(resto_f, escolha, N_ITENS_VALIDOS_FIS)
    df = pd.DataFrame(pct, columns=ALTERNATIVAS)
    df.insert(0, "Acertos_demais_fisica", np.arange(N_ITENS_VALIDOS_FIS))
    df.insert(0, "Participantes", n)
    df.insert(0, "Questao", q)
    curvas_fisica.append(df)

curvas_tri = pd.concat(curvas_tri, ignore_index=True)
curvas_acertos = pd.concat(curvas_acertos, ignore_index=True)
curvas_fisica = pd.concat(curvas_fisica, ignore_index=True)
curvas = curvas_tri   # nome mantido para compatibilidade com etapas seguintes

del resp_azul, acertos; gc.collect()

# ---------- Gráficos ----------
def desenhar(ax, dados, eixo_x, gab, titulo, legenda=True):
    dados = dados.loc[dados["Participantes"] >= FAIXA_MIN_PARTICIPANTES]
    for alt in ALTERNATIVAS:
        if alt == gab:
            ax.plot(dados[eixo_x], dados[alt], color="black", lw=2.5, label=f"{alt} (gabarito)")
        else:
            ax.plot(dados[eixo_x], dados[alt], lw=1.3, marker="o", ms=2.5, label=alt)
    ax.set_title(titulo, fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    if legenda:
        ax.legend(fontsize=6, loc="upper left")

reguas = [
    ("tri", curvas_tri, "Nota_media_faixa", f"Nota TRI em CN ({COL_PROFICIENCIA}, média da faixa)"),
    ("acertos", curvas_acertos, "Acertos_demais_itens",
     f"Acertos nos demais itens válidos de CN (0 a {N_ITENS_VALIDOS_CN - 1})"),
    ("fisica", curvas_fisica, "Acertos_demais_fisica",
     f"Acertos nas demais questões de Física (0 a {N_ITENS_VALIDOS_FIS - 1})"),
]
TITULOS = {"tri": "Régua: nota TRI", "acertos": "Régua: acertos nos demais itens de CN",
           "fisica": "Régua: acertos nas demais de Física"}

n = len(validas); ncol = 4; nlin = int(np.ceil(n / ncol))
for nome, dados_curva, eixo_x, rotulo_x in reguas:
    fig, axes = plt.subplots(nlin, ncol, figsize=(4 * ncol, 3.2 * nlin), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, it) in zip(axes, validas.iterrows()):
        q, gab = int(it["Questao"]), it["Gabarito"]
        desenhar(ax, dados_curva.loc[dados_curva["Questao"].eq(q)], eixo_x, gab,
                 f"Questão {q} — gabarito {gab}")
    for ax in axes[n:]:
        ax.axis("off")
    fig.supxlabel(rotulo_x)
    fig.supylabel("% que escolheu a alternativa")
    plt.tight_layout()
    plt.savefig(os.path.join(PASTA_FIG, f"curvas_distratores_{nome}.png"), dpi=300, bbox_inches="tight")
    plt.show()

# Uma figura por questão, com as três réguas lado a lado (para o TCC)
for _, it in validas.iterrows():
    q, gab = int(it["Questao"]), it["Gabarito"]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
    for ax, (nome, dados_curva, eixo_x, rotulo_x) in zip(axes, reguas):
        desenhar(ax, dados_curva.loc[dados_curva["Questao"].eq(q)], eixo_x, gab,
                 TITULOS[nome])
        ax.set_xlabel(rotulo_x)
    axes[0].set_ylabel("% de escolha")
    fig.suptitle(f"ENEM 2024 — Questão {q} (caderno azul), gabarito {gab}")
    plt.tight_layout()
    plt.savefig(os.path.join(PASTA_FIG, f"curva_Q{q}.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

# Planilha própria com as três versões das curvas (não depende da Etapa 7)
ARQ_CURVAS = os.path.join(PASTA_SAIDA, "curvas_distratores_ENEM2024.xlsx")
with pd.ExcelWriter(ARQ_CURVAS) as w:
    curvas_tri.to_excel(w, sheet_name="Regua 1 - nota TRI", index=False)
    curvas_acertos.to_excel(w, sheet_name="Regua 2 - acertos CN", index=False)
    curvas_fisica.to_excel(w, sheet_name="Regua 3 - acertos Fisica", index=False)
print("Planilha das curvas:", ARQ_CURVAS)

print("Figuras salvas em:", PASTA_FIG)
print("  curvas_distratores_tri.png      — todas as questões, régua nota TRI")
print("  curvas_distratores_acertos.png  — todas as questões, régua acertos nos demais itens")
print("  curvas_distratores_fisica.png   — todas as questões, régua acertos nas demais de Física")
print("  curva_Q91.png ... curva_Q132.png — uma por questão, as três réguas lado a lado")


In [ ]:
# ============================================================
# ETAPA 6 — PARÂMETROS DA TRI (a, b, c) × TAXA DE ACERTO
# ============================================================
# Modelo do Enem (INEP, Procedimentos de Análise, 2021, p. 15):
#   P(acerto | θ) = c + (1 − c) / (1 + exp[−a(θ − b)])      (sem constante D)
# Parâmetros na escala (0,1); nota = 100·θ + 500 (mesma fonte, p. 20).
#
# Para cada questão:
#   • b convertido para a escala da nota (100·b + 500);
#   • taxa de acerto OBSERVADA nos participantes com nota válida;
#   • taxa de acerto PREVISTA pela TRI = média de P(acerto | θ) desses
#     mesmos participantes, usando θ = (NU_NOTA_CN − 500) / 100;
#   • diferença observada − prevista (em pontos percentuais).

from scipy.stats import spearmanr

nota = respostas_fisica[COL_PROFICIENCIA].to_numpy(dtype="float64")
com_nota = np.isfinite(nota) & (nota > 0)
theta = (nota[com_nota] - 500) / 100
print(f"Participantes com nota válida em CN: {com_nota.sum():,}")

linhas_tri = []
for _, it in validas.iterrows():
    q = int(it["Questao"])
    a, b, c = float(it["Param_a"]), float(it["Param_b"]), float(it["Param_c"])
    if not np.isfinite([a, b, c]).all():
        continue
    obs = 100 * np.mean(respostas_fisica[f"Q{q}"].to_numpy()[com_nota] == it["Gabarito"])
    prev = 100 * np.mean(c + (1 - c) / (1 + np.exp(-a * (theta - b))))
    linhas_tri.append({
        "Questao": q, "Gabarito": it["Gabarito"],
        "a": a, "b": b, "c": c,
        "b_escala_nota": 100 * b + 500,
        "Acerto_observado_%": obs,
        "Acerto_previsto_TRI_%": prev,
        "Diferenca_obs_menos_prev_pp": obs - prev,
        "Acerto_ao_acaso_%": 100 * c,
    })

tabela_tri = pd.DataFrame(linhas_tri).sort_values("Questao").reset_index(drop=True)

rho_b, p_b = spearmanr(tabela_tri["b"], tabela_tri["Acerto_observado_%"])
rho_prev, p_prev = spearmanr(tabela_tri["Acerto_previsto_TRI_%"], tabela_tri["Acerto_observado_%"])
r_prev = np.corrcoef(tabela_tri["Acerto_previsto_TRI_%"], tabela_tri["Acerto_observado_%"])[0, 1]

print(f"\nQuestões analisadas: {len(tabela_tri)}")
print(f"Spearman b × acerto observado:          {rho_b:.3f} (p = {p_b:.4f})")
print(f"Spearman previsto TRI × observado:       {rho_prev:.3f} (p = {p_prev:.4f})")
print(f"Pearson  previsto TRI × observado:       {r_prev:.3f}")
print(f"Diferença média |obs − prev|:            {tabela_tri['Diferenca_obs_menos_prev_pp'].abs().mean():.2f} p.p.")

display(tabela_tri.style.format({
    "a": "{:.2f}", "b": "{:.2f}", "c": "{:.2f}", "b_escala_nota": "{:.0f}",
    "Acerto_observado_%": "{:.1f}", "Acerto_previsto_TRI_%": "{:.1f}",
    "Diferenca_obs_menos_prev_pp": "{:+.1f}", "Acerto_ao_acaso_%": "{:.0f}",
}))

# ---------- Gráfico 1: b × acerto, cor = c, tamanho = a ----------
fig, ax = plt.subplots(figsize=(9, 6))
pts = ax.scatter(tabela_tri["b_escala_nota"], tabela_tri["Acerto_observado_%"],
                 c=tabela_tri["c"], cmap="viridis", s=60 + 60 * tabela_tri["a"],
                 edgecolor="black", linewidth=0.6)
for _, r in tabela_tri.iterrows():
    ax.annotate(f"Q{int(r['Questao'])}", (r["b_escala_nota"], r["Acerto_observado_%"]),
                textcoords="offset points", xytext=(7, 5), fontsize=8)
ax.axhline(20, color="gray", ls=":", lw=1)
ax.text(ax.get_xlim()[0], 20.5, " 20% = chute puro (1 em 5)", fontsize=7, color="gray", va="bottom")
cb = fig.colorbar(pts, ax=ax)
cb.set_label("Parâmetro c (acerto ao acaso)")
ax.set_xlabel("Dificuldade b na escala da nota (100·b + 500)")
ax.set_ylabel("Taxa de acerto observada (%)")
ax.set_title(f"Dificuldade TRI × acerto — Física ENEM 2024 (Spearman = {rho_b:.2f})\n"
             "cor = parâmetro c; tamanho do ponto = parâmetro a")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIG, "tri_b_vs_acerto.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---------- Gráfico 2: previsto pela TRI × observado ----------
fig, ax = plt.subplots(figsize=(7, 7))
lim = [0, max(tabela_tri[["Acerto_observado_%", "Acerto_previsto_TRI_%"]].max()) + 5]
ax.plot(lim, lim, color="gray", ls="--", lw=1, label="previsto = observado")
ax.scatter(tabela_tri["Acerto_previsto_TRI_%"], tabela_tri["Acerto_observado_%"],
           s=50, edgecolor="black", linewidth=0.6, zorder=3)
for _, r in tabela_tri.iterrows():
    ax.annotate(f"Q{int(r['Questao'])}", (r["Acerto_previsto_TRI_%"], r["Acerto_observado_%"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Taxa de acerto prevista pelo modelo da TRI (%)")
ax.set_ylabel("Taxa de acerto observada (%)")
ax.set_title("Acerto previsto pela TRI × observado — Física ENEM 2024")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIG, "tri_previsto_vs_observado.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---------- Planilha própria (não depende da Etapa 7) ----------
ARQ_TRI = os.path.join(PASTA_SAIDA, "tri_parametros_ENEM2024.xlsx")
with pd.ExcelWriter(ARQ_TRI) as w:
    tabela_tri.to_excel(w, sheet_name="Parametros e acerto", index=False)
    pd.DataFrame({
        "Medida": ["Questões analisadas",
                   "Spearman b × acerto observado", "p-valor",
                   "Spearman previsto × observado", "p-valor",
                   "Pearson previsto × observado",
                   "Diferença média absoluta obs − prev (p.p.)",
                   "Modelo", "Escala"],
        "Valor": [len(tabela_tri), rho_b, p_b, rho_prev, p_prev, r_prev,
                  tabela_tri["Diferenca_obs_menos_prev_pp"].abs().mean(),
                  "P = c + (1 − c)/(1 + exp[−a(θ − b)]) (INEP, 2021)",
                  "θ = (NU_NOTA_CN − 500)/100; b na nota = 100·b + 500"],
    }).to_excel(w, sheet_name="Resumo", index=False)
print("\nPlanilha:", ARQ_TRI)
print("Figuras: tri_b_vs_acerto.png, tri_previsto_vs_observado.png")


In [ ]:
# ============================================================
# ETAPA 7 — EXPORTAÇÃO
# ============================================================
ARQ_XLSX = os.path.join(PASTA_SAIDA, "analise_fisica_ENEM2024.xlsx")
with pd.ExcelWriter(ARQ_XLSX) as w:
    tabela_distratores.to_excel(w, sheet_name="Distratores por questão", index=False)
    estatisticas.to_excel(w, sheet_name="Estatísticas")
    curvas.to_excel(w, sheet_name="Curvas por faixa", index=False)
    fisica.to_excel(w, sheet_name="Itens e TRI", index=False)
    pd.DataFrame({
        "Etapa": ["Participantes analisados", "Cadernos usados", "Questões selecionadas",
                  "Questões válidas", "Variável de proficiência", "Faixas de proficiência"],
        "Valor": [N_PARTICIPANTES, ", ".join(map(str, sorted(MAPA_COR_PARA_AZUL))),
                  len(fisica), len(validas), COL_PROFICIENCIA, N_FAIXAS],
    }).to_excel(w, sheet_name="Resumo", index=False)

tabela_distratores.to_csv(os.path.join(PASTA_SAIDA, "distratores_fisica_ENEM2024.csv"),
                          sep=";", index=False, encoding="utf-8-sig")
print("Gerado:", ARQ_XLSX)
for f in sorted(os.listdir(PASTA_FIG)):
    print("  figura:", f)